# Machine Translation using Transformer

Train a Transformer model on French-English translation task using the fra-eng dataset.

In [ ]:
import sys
import os
sys.path.insert(0, '../..')
os.chdir('../..')

import torch
from utils import spy

## 1. Load and preprocess data

In [ ]:
# Load fra-eng dataset using MachineTranslation class
data = spy.MachineTranslation(
    path='data/fra-eng/fra.txt',
    batch_size=32,
    num_steps=10,
    num_train=1000,  # Use 1000 training examples
    num_val=200      # Use 200 validation examples
)

print(f"Source vocabulary size: {len(data.src_vocab)}")
print(f"Target vocabulary size: {len(data.tgt_vocab)}")
print(f"Source vocab sample: {data.src_vocab.idx_to_token[:10]}")
print(f"Target vocab sample: {data.tgt_vocab.idx_to_token[:10]}")

## 2. Define Transformer model

In [ ]:
# Create Transformer encoder and decoder
src_vocab_size = len(data.src_vocab)
tgt_vocab_size = len(data.tgt_vocab)
num_hiddens, ffn_num_hiddens, num_heads, num_blks = 256, 512, 4, 2
dropout = 0.1

encoder = spy.TransformerEncoder(
    vocab_size=src_vocab_size,
    num_hiddens=num_hiddens,
    ffn_num_hiddens=ffn_num_hiddens,
    num_heads=num_heads,
    num_blks=num_blks,
    dropout=dropout
)

decoder = spy.TransformerDecoder(
    vocab_size=tgt_vocab_size,
    num_hiddens=num_hiddens,
    ffn_num_hiddens=ffn_num_hiddens,
    num_heads=num_heads,
    num_blks=num_blks,
    dropout=dropout
)

# Create EncoderDecoder model
model = spy.EncoderDecoder(encoder, decoder)
model.lr = 0.001

## 3. Create custom training class for seq2seq

In [ ]:
class Seq2SeqTrainer(spy.Trainer):
    """Custom trainer for sequence-to-sequence tasks"""
    
    def fit_epoch(self):
        """Override fit_epoch to handle seq2seq training"""
        self.model.train()
        train_total, train_batches = 0.0, 0
        
        for batch in self.train_loader:
            # batch: (src, tgt_input, src_valid_len, tgt_output)
            batch = [b.to(self.device) for b in batch]
            src, tgt_input, src_valid_len, tgt_output = batch
            
            # Forward pass
            Y_hat = self.model(src, tgt_input, src_valid_len)
            
            # Compute loss (reshape for cross-entropy)
            loss = torch.nn.functional.cross_entropy(
                Y_hat.reshape(-1, Y_hat.shape[-1]),
                tgt_output.reshape(-1),
                reduction='mean'
            )
            
            # Backward pass
            self.optimizer.zero_grad()
            loss.backward()
            self._clip_gradients()
            self.optimizer.step()
            
            train_total += loss.item()
            train_batches += 1
        
        train_loss = train_total / train_batches if train_batches else 0.0
        
        # Validation
        val_loss = None
        if self.val_loader is not None:
            self.model.eval()
            with torch.no_grad():
                val_total, val_batches = 0.0, 0
                for batch in self.val_loader:
                    batch = [b.to(self.device) for b in batch]
                    src, tgt_input, src_valid_len, tgt_output = batch
                    
                    Y_hat = self.model(src, tgt_input, src_valid_len)
                    loss = torch.nn.functional.cross_entropy(
                        Y_hat.reshape(-1, Y_hat.shape[-1]),
                        tgt_output.reshape(-1),
                        reduction='mean'
                    )
                    
                    val_total += loss.item()
                    val_batches += 1
                
                val_loss = val_total / val_batches if val_batches else None
        
        return train_loss, val_loss, None
    
    def fit(self, model, data):
        """Override fit to use custom training loop"""
        self.prepare_data(data)
        self.prepare_model(model)
        
        # Initialize board
        if 'train_loss' not in model.board:
            model.board['train_loss'] = []
        if 'val_loss' not in model.board:
            model.board['val_loss'] = []
        
        for epoch in range(self.max_epochs):
            train_loss, val_loss, _ = self.fit_epoch()
            model.board['train_loss'].append(train_loss)
            if val_loss is not None:
                model.board['val_loss'].append(val_loss)
            
            print(f"Epoch {epoch+1}/{self.max_epochs}, "
                  f"train_loss: {train_loss:.4f}, "
                  f"val_loss: {val_loss:.4f}" if val_loss else f"train_loss: {train_loss:.4f}")
            
            model.plot()
        
        if model._fig is not None:
            import matplotlib.pyplot as plt
            plt.close(model._fig)

## 4. Train the model

In [ ]:
# Create trainer and train the model
trainer = Seq2SeqTrainer(max_epochs=10, gradient_clip_val=1.0)
trainer.fit(model, data)

## 5. Test translation (optional)

In [ ]:
def translate(src_sentence, model, data):
    """Translate a single source sentence"""
    # Preprocess source sentence
    src_tokens = src_sentence.split()
    src_sentence_processed = data._preprocess(' '.join(src_tokens) + ' <eos>')
    src_tokens_processed = [t for t in src_sentence_processed.split() if t]
    
    # Pad or trim
    pad_or_trim = lambda seq, t: (
        seq[:t] if len(seq) > t else seq + ['<pad>'] * (t - len(seq))
    )
    src_tokens_padded = pad_or_trim(src_tokens_processed, data.num_steps)
    
    # Convert to tensor
    src_indices = torch.tensor(data.src_vocab[src_tokens_padded]).unsqueeze(0)
    src_valid_len = torch.tensor([len(src_tokens_processed)])
    
    # Inference
    model.eval()
    with torch.no_grad():
        enc_outputs = model.encoder(src_indices.to(trainer.device), src_valid_len)
        dec_state = model.decoder.init_state(enc_outputs, src_valid_len)
        
        tgt_tokens = ['<bos>']
        for _ in range(data.num_steps):
            tgt_idx = torch.tensor([data.tgt_vocab[tgt_tokens[-1]]]).unsqueeze(0).to(trainer.device)
            Y_hat, dec_state = model.decoder(tgt_idx, dec_state)
            Y_hat = Y_hat.squeeze(0)
            
            # Get next token
            pred_idx = Y_hat.argmax(dim=-1, keepdim=True).item()
            pred_token = data.tgt_vocab.to_tokens(pred_idx)
            
            if pred_token == '<eos>':
                break
            tgt_tokens.append(pred_token)
    
    return ' '.join(tgt_tokens[1:])

# Evaluate on validation set using BLEU score
def evaluate_bleu(model, data, num_examples=50):
    """Evaluate model using BLEU score on validation set"""
    bleu_scores = []
    translations = []
    
    val_loader = data.val_dataloader()
    model.eval()
    
    for i, batch in enumerate(val_loader):
        if i >= num_examples:
            break
        
        batch = [b.to(trainer.device) for b in batch]
        src, tgt_input, src_valid_len, tgt_output = batch
        
        # Get model predictions
        with torch.no_grad():
            Y_hat = model(src, tgt_input, src_valid_len)
            pred_tokens = Y_hat.argmax(dim=-1)
        
        # Convert to actual tokens and compute BLEU
        batch_size = src.shape[0]
        for j in range(batch_size):
            # Reference (ground truth)
            ref = [data.tgt_vocab.to_tokens(idx.item()) for idx in tgt_output[j] 
                   if idx.item() != data.tgt_vocab['<pad>']]
            
            # Prediction
            pred = [data.tgt_vocab.to_tokens(idx.item()) for idx in pred_tokens[j]
                    if idx.item() != data.tgt_vocab['<pad>']]
            
            # Compute BLEU (4-gram)
            bleu = spy.bleu(pred, ref, k=4)
            bleu_scores.append(bleu)
            translations.append({
                'source': data.src_vocab.to_tokens(src[j].cpu().tolist()),
                'reference': ref,
                'prediction': pred,
                'bleu': bleu
            })
    
    avg_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0
    return avg_bleu, translations

# Evaluate and show results
print("=" * 80)
print("Evaluating on validation set...")
print("=" * 80)
avg_bleu, translations = evaluate_bleu(model, data, num_examples=10)

print(f"\nAverage BLEU score: {avg_bleu:.4f}\n")
print("=" * 80)
print("Sample translations:")
print("=" * 80)

for i, trans in enumerate(translations[:10]):
    print(f"\n[Example {i+1}]")
    src_str = ' '.join(trans['source']).replace('<pad>', '').strip()
    ref_str = ' '.join(trans['reference'])
    pred_str = ' '.join(trans['prediction'])
    
    print(f"Source:      {src_str}")
    print(f"Reference:   {ref_str}")
    print(f"Prediction:  {pred_str}")
    print(f"BLEU score:  {trans['bleu']:.4f}")